# YOLOX Plate Detector Training (Official Repo + CUDA)

This notebook clones the official YOLOX repository, installs a CUDA-enabled PyTorch, installs YOLOX in editable mode, materializes a tuned experiment for Indonesian license plates, runs training (FP16), and exports ONNX.

Requirements:
- Kaggle Notebook with GPU enabled (T4/A100).
- Internet ON for `pip` and `git`.
- Attach your dataset via ‘Add data’.


In [ ]:
# --- Configuration ---
from pathlib import Path

# Paths to your Kaggle datasets
DATA_DIR = Path('/kaggle/input/your-dataset-slug')  # change me
TRAIN_JSON = 'annotations/instances_Train.json'
VAL_JSON = 'annotations/instances_Validation.json'
TRAIN_NAME = 'images/Train'
VAL_NAME = 'images/Validation'

# Optional: COCO pretrain checkpoint (attach as dataset)
PRETRAIN_CKPT = Path('/kaggle/input/yolox-weights/YOLOX_S.pth')  # or '' to train from scratch

# Training hyperparameters
EXPN = 'plate_yolox_s'
EPOCHS = 80
NO_AUG_EPOCHS = 15
BATCH = 16
FP16 = True

EXPORT_DIR = Path('/kaggle/working/export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Config OK')


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Install CUDA PyTorch + YOLOX + deps
import sys, subprocess

def run(cmd):
    print('>>>', ' '.join(cmd))
    subprocess.check_call(cmd)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
# Try cu121 first, fallback to cu118 if unavailable in the image
try:
    run([sys.executable, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch', 'torchvision', 'torchaudio'])
except subprocess.CalledProcessError:
    run([sys.executable, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cu118', 'torch', 'torchvision', 'torchaudio'])

# Clone official YOLOX and install editable
run(['git', 'clone', 'https://github.com/Megvii-BaseDetection/YOLOX.git', '/kaggle/working/YOLOX'])
run([sys.executable, '-m', 'pip', 'install', '-e', '/kaggle/working/YOLOX'])

# Extra tooling
run([sys.executable, '-m', 'pip', 'install', 'pycocotools', 'onnx', 'onnxsim'])

import torch as _t
print('Torch:', _t.__version__, 'CUDA:', getattr(_t.version, 'cuda', None), 'is_available:', _t.cuda.is_available())
if _t.cuda.is_available():
    print('Device:', _t.cuda.get_device_name(0))


In [ ]:
# Materialize tuned experiment file
exp_text = r'''
"""YOLOX-s experiment for Indonesian license plate detection.

This experiment targets a single class (license_plate) and allows runtime
override of dataset paths via environment variables so you can train without
moving data around. Defaults are repo-local but can be changed at run time.

Env overrides (optional):
- YOLOX_DATA_DIR   -> base data directory
- YOLOX_TRAIN_ANN  -> path to train annotation JSON (relative to DATA_DIR or absolute)
- YOLOX_VAL_ANN    -> path to val annotation JSON (relative to DATA_DIR or absolute)
- YOLOX_TRAIN_NAME -> subfolder name for train images (default: train)
- YOLOX_VAL_NAME   -> subfolder name for val images (default: val)
"""
from __future__ import annotations
import os
from yolox.exp import Exp as _BaseExp

class Exp(_BaseExp):
    def __init__(self) -> None:
        super().__init__()
        self.depth = 0.33
        self.width = 0.50
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.num_classes = 1
        self.max_epoch = int(os.getenv('YOLOX_MAX_EPOCH', '80'))
        self.no_aug_epochs = int(os.getenv('YOLOX_NO_AUG', '15'))
        self.warmup_epochs = 3
        self.basic_lr_per_img = 0.01 / 64.0
        self.eval_interval = 1
        self.print_interval = 50
        self.data_num_workers = 4
        self.random_size = (14, 26)
        self.mosaic_prob = 0.7
        self.mixup_prob = 0.05
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.enable_mixup = True
        self.use_l1 = True
        data_dir = os.getenv('YOLOX_DATA_DIR', 'data/yolox/cam01')
        train_ann = os.getenv('YOLOX_TRAIN_ANN', 'annotations/train.json')
        val_ann = os.getenv('YOLOX_VAL_ANN', 'annotations/val.json')
        train_name = os.getenv('YOLOX_TRAIN_NAME', 'train')
        val_name = os.getenv('YOLOX_VAL_NAME', 'val')
        self.data_dir = data_dir
        self.train_ann = train_ann
        self.val_ann = val_ann
        self.train_name = train_name
        self.val_name = val_name
        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split('.')[0]
        self.pretrained = os.getenv('YOLOX_PRETRAIN', '')
'''
exp_path = Path('/kaggle/working/exp_plate_yolox_s.py')
exp_path.write_text(exp_text, encoding='utf-8')
print('Wrote:', exp_path)


In [ ]:
# Train
import os, subprocess, sys
env = os.environ.copy()
env['YOLOX_DATA_DIR'] = str(DATA_DIR)
env['YOLOX_TRAIN_ANN'] = str(TRAIN_JSON)
env['YOLOX_VAL_ANN'] = str(VAL_JSON)
env['YOLOX_TRAIN_NAME'] = str(TRAIN_NAME)
env['YOLOX_VAL_NAME'] = str(VAL_NAME)
env['YOLOX_MAX_EPOCH'] = str(EPOCHS)
env['YOLOX_NO_AUG'] = str(NO_AUG_EPOCHS)
if PRETRAIN_CKPT and str(PRETRAIN_CKPT) != '':
    env['YOLOX_PRETRAIN'] = str(PRETRAIN_CKPT)

cmd = [sys.executable, '-m', 'yolox.tools.train', '-f', str('/kaggle/working/exp_plate_yolox_s.py'), '-d', '1', '-b', str(BATCH), '--expn', EXPN, '-o']
if FP16:
    cmd.append('--fp16')
if PRETRAIN_CKPT and str(PRETRAIN_CKPT) != '':
    cmd.extend(['-c', str(PRETRAIN_CKPT)])
print('Launching training:', ' '.join(cmd))
subprocess.check_call(cmd, env=env)


In [ ]:
# Evaluate best checkpoint and export ONNX + metrics
import json, shutil, re, subprocess, sys
from pathlib import Path

out_dir = Path(f'/kaggle/working/YOLOX_outputs/{EXPN}')
ckpt = out_dir / 'best_ckpt.pth'
assert ckpt.exists(), f'Missing checkpoint: {ckpt}'

# Run YOLOX eval to get fresh COCO metrics from best checkpoint
cmd_eval = [sys.executable, '-m', 'yolox.tools.eval', '-f', str('/kaggle/working/exp_plate_yolox_s.py'), '-c', str(ckpt)]
print('Evaluating checkpoint:', ' '.join(cmd_eval))
proc = subprocess.run(cmd_eval, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)

# Parse COCO summary from stdout
metrics = {}
pat_map = re.compile(r'Average Precision\s*\(AP\) @\[ IoU=0.50:0.95 \]\s*=\s*([0-9.]+)')
pat_ap50 = re.compile(r'Average Precision\s*\(AP\) @ IoU=0.50\s*=\s*([0-9.]+)')
pat_ap75 = re.compile(r'Average Precision\s*\(AP\) @ IoU=0.75\s*=\s*([0-9.]+)')
pat_aps = re.compile(r'Average Precision\s*\(AP\) small\s*=\s*([0-9.]+)')
pat_apm = re.compile(r'Average Precision\s*\(AP\) medium\s*=\s*([0-9.]+)')
pat_apl = re.compile(r'Average Precision\s*\(AP\) large\s*=\s*([0-9.]+)')

def grab(p):
    m = p.search(proc.stdout)
    return float(m.group(1)) if m else None

metrics['AP'] = grab(pat_map)
metrics['AP50'] = grab(pat_ap50)
metrics['AP75'] = grab(pat_ap75)
metrics['APS'] = grab(pat_aps)
metrics['APM'] = grab(pat_apm)
metrics['APL'] = grab(pat_apl)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
with (EXPORT_DIR / 'metrics.json').open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Wrote metrics.json with keys:', [k for k,v in metrics.items() if v is not None])

# Export ONNX
onnx_out = EXPORT_DIR / 'yolox_s.onnx'
cmd = [sys.executable, '-m', 'yolox.tools.export', '-f', str('/kaggle/working/exp_plate_yolox_s.py'), '-c', str(ckpt), '--export', 'onnx', '--decode_in_inference']
print('Exporting ONNX:', ' '.join(cmd))
subprocess.check_call(cmd)

# Move ONNX if created under working dir
generated = Path('/kaggle/working') / 'model.onnx'
if generated.exists():
    shutil.move(str(generated), str(onnx_out))
print('ONNX at', onnx_out)

# Copy best checkpoint
shutil.copy2(str(ckpt), str(EXPORT_DIR / 'best_ckpt.pth'))

print('Export dir contents:', list(EXPORT_DIR.iterdir()))
